# Query and export template

Use a prepared direction-aware dataset. Set `DATA_PATH` to a local CSV or Parquet file created with JapanTrade.

In [ ]:
from pathlib import Path
import tempfile

from japantrade.analytics import load_normalized_data
from japantrade.exporters import EXAMPLE_QUERIES, export_to_duckdb, export_to_sqlite, run_example_query

DATA_PATH = Path('data/japan_exports_2023_2025.parquet')
df = load_normalized_data(DATA_PATH)
df[['direction', 'kind', 'country', 'code', 'date', 'unit', 'value']].head()

## Export a queryable copy

DuckDB and SQLite exports are useful when an analyst needs ad-hoc SQL. The `direction` column remains part of the dataset.

In [ ]:
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)
duckdb_path = export_to_duckdb(df, output_dir / 'trade.duckdb')
sqlite_path = export_to_sqlite(df, output_dir / 'trade.sqlite')
duckdb_path, sqlite_path

In [ ]:
params = {
    'start_date': df['date'].min().strftime('%Y-%m-%d'),
    'end_date': df['date'].max().strftime('%Y-%m-%d'),
    'kind': 'HS',
    'limit': 10,
    'year': df['date'].max().year,
}

for query in EXAMPLE_QUERIES:
    print(query.name, '-', query.description)
    display(run_example_query('duckdb', duckdb_path, query.name, params).head())